# Module C, sequence 2: saving and loading (corrected version)

In [ ]:
import numpy as np
from pathlib import Path
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.data import Axis, DataRaw, DataToExport
from pymodaq_data.h5modules.saving import H5SaverLowLevel
from pymodaq_data.h5modules.data_saving import DataToExportSaver, DataLoader

DATA = Path('../data')

wavelength = Axis('wavelength', 'nm', data=np.linspace(500, 560, 121), index=0)
signal = 3 * gauss1D(wavelength.get_data(), 528, 4) + 0.1
line = DataRaw('line', data=[signal], axes=[wavelength], labels=['photodiode'], units='V')
line.errors = [np.full(121, 0.05)]
mesure = DataToExport('mesure', data=[line])

## 1. Write the file

In [ ]:
path = Path('line.h5')
path.unlink(missing_ok=True)  # init_file appends to an existing file

h5saver = H5SaverLowLevel()
h5saver.init_file(path)
h5saver.add_det_group('/RawData', 'Photodiode line')
DataToExportSaver(h5saver).add_data('/RawData/Detector000', mesure)
for node in h5saver.walk_nodes('/'):
    print(node)
h5saver.close_file()

## 2. Read it back and compare

In [ ]:
loader = DataLoader(path)
loaded = loader.load_all('/RawData/Detector000')
loader.close_file()
back = loaded.get_data_from_name('line')
print(back == line, back.axes, back.errors[0][:3])

## 3. Read the continuous recording of the DAQ_Viewer (enlargeable arrays)

In [ ]:
loader = DataLoader(DATA / 'continuous_spectra.h5')
for node in loader.walk_nodes('/'):
    print(node)
spectra = loader.load_data('/RawData/Detector000/Data1D/CH00/EnlData00', load_all=True)
loader.close_file()
print(spectra.shape[0], 'spectra recorded, shape', spectra.shape)
timestamps = spectra.get_axis_from_index(0)[0]
print(timestamps.label, timestamps.units, timestamps.get_data()[:3])

## 4. Mean and standard deviation of the series

In [ ]:
mean_spectrum = spectra.mean(0)
std_spectrum = spectra.data[0].std(axis=0, ddof=1)
wavelength = spectra.get_axis_from_index(1)[0].get_data()
i_peak = int(np.argmax(mean_spectrum.data[0]))
print(wavelength[i_peak], mean_spectrum.data[0][i_peak], std_spectrum[i_peak])